# Lesson 25 — Customer Segmentation
## AI-Powered Customer Retention System

---

### Overview & Objectives
In this lesson, we perform **Customer Segmentation** on the Brazilian E-Commerce (Olist) dataset. Customer segmentation is the practice of dividing a customer base into distinct groups that share similar purchase behaviors, transaction frequencies, and monetary value.

By segmenting customers, the business can:
- Identify VIP / high-value customers for loyalty programs.
- Detect at-risk customers early and trigger proactive retention campaigns.
- Nurture new buyers into repeat customers.
- Optimize marketing spend by tailoring campaigns to specific behavioral clusters.

### Lesson 25 Complete Roadmap:
1. **Section 1 — Environment Setup & Database Connection**: Connect Python to the PostgreSQL `customer_retention` database.
2. **Section 2 — Data Loading**: Load `customers`, `orders`, `order_items`, and `order_payments` from PostgreSQL.
3. **Section 3 — Customer-Level Dataset**: Aggregate orders and spending at the `customer_unique_id` level.
4. **Section 4 — RFM Metrics Calculation**: Calculate Recency (days), Frequency (orders), and Monetary (spending).
5. **Section 5 — Check RFM Data**: Validate values and remove non-positive transactions.
6. **Section 6 — RFM Scoring**: Assign quintile scores (1 to 5) for R, F, and M, and numeric RFM Score.
7. **Section 7 — Rule-Based RFM Segmentation**: Assign business rule labels (*High Value*, *Loyal*, *At Risk*, *New*, *Regular*).
8. **Section 8 — Feature Preparation for Machine Learning**: Log-transform skewed RFM features and scale via `StandardScaler`.
9. **Section 9 — K-Means Clustering Experiments**: Train K-Means for $K \in \{2, 3, 4, 5, 6\}$.
10. **Section 10 — Visualize Cluster Metrics**: Plot Elbow Curve (Inertia) and Silhouette Scores.
11. **Section 11 — Optimal Cluster Selection**: Dynamically select optimal $K$ based on highest Silhouette Score.
12. **Section 12 — Final K-Means Model**: Train the final model and assign cluster labels.
13. **Section 13 — Cluster Profiling**: Statistical summaries (mean, count, percentage) of RFM per cluster.
14. **Section 14 — Relative Cluster Interpretation**: Profile-based behavioral labeling relative to population averages.
15. **Section 15 — Visualizations**: Matplotlib plots for distribution, spend, and RFM scatter spaces.
16. **Section 16 — Dataset Export & Quality Checks**: Export to `notebooks/data/customer_segments.csv` and run validation tests.
17. **Section 17 — Business Insights**: Strategic retention takeaways and application guide.

---
## Section 1: Environment Setup & Database Connection

In this section, we import the required data science, machine learning, and database connectivity libraries. We then securely establish a connection to the PostgreSQL database (`customer_retention`) using environment variables with SQLAlchemy and verify the connection by inspecting available tables.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 1: LIBRARIES IMPORT
# =========================================================

import os

# Data manipulation and numerical operations
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt

# Database connectivity
from sqlalchemy import create_engine, text

# Machine learning for clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Display and plot formatting
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

print("Libraries imported successfully!")
print("=" * 60)
print(f"Pandas version : {pd.__version__}")
print(f"NumPy version  : {np.__version__}")
print("=" * 60)

### Connect to PostgreSQL Database (`customer_retention`)

We create a connection engine using SQLAlchemy. We retrieve database credentials securely from environment variables (`POSTGRES_PASSWORD`). We then execute a lightweight, read-only query against `information_schema.tables` to verify that our connection is active and list the available tables.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 1: POSTGRESQL DATABASE CONNECTION
# =========================================================

import os

# Database connection credentials from environment variables
DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "customer_retention")

# Ensure required password environment variable is set
if not DB_PASSWORD:
    raise ValueError(
        "Missing database password: The 'POSTGRES_PASSWORD' environment variable is not set. "
        "Please set the POSTGRES_PASSWORD environment variable before running this notebook."
    )

# Create SQLAlchemy connection engine
DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
engine = create_engine(DATABASE_URL)

# Verify connection and inspect existing database tables (Read-Only)
try:
    with engine.connect() as connection:
        query = text(
            """
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public' 
            ORDER BY table_name;
            """
        )
        result = connection.execute(query)
        tables = [row[0] for row in result.fetchall()]
        
    print("Database connection established successfully!")
    print("=" * 60)
    print(f"Connected to database : {DB_NAME}")
    print(f"Host / Port            : {DB_HOST}:{DB_PORT}")
    print("=" * 60)
    print(f"Found {len(tables)} tables in database:")
    for table in tables:
        print(f"  • {table}")
    print("=" * 60)
except Exception as e:
    print(f"Connection failed: {e}")

---
## Section 2: Load Required Tables from PostgreSQL

In this section, we extract the core transactional and customer tables required for customer segmentation using read-only SQL queries via the existing SQLAlchemy engine:
1. **`customers`**: Maps each order-level `customer_id` to the persistent `customer_unique_id` representing the real person/buyer.
2. **`orders`**: Contains purchase timestamps and order statuses (filtered for `delivered` orders).
3. **`order_items`**: Contains item prices and freight charges.
4. **`order_payments`**: Contains payment transaction values.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 2: LOAD DATA FROM POSTGRESQL
# =========================================================

# 1. Load customers table
customers = pd.read_sql(
    "SELECT customer_id, customer_unique_id FROM customers;", 
    engine
)
print("CUSTOMERS TABLE LOADED")
print("=" * 60)
print(f"Rows: {len(customers):,}, Columns: {customers.shape[1]}")
print(f"Columns: {customers.columns.tolist()}")
display(customers.head(3))
print("=" * 60)

# 2. Load orders table (filtered for delivered orders)
orders = pd.read_sql(
    """
    SELECT order_id, customer_id, order_status, order_purchase_timestamp 
    FROM orders 
    WHERE order_status = 'delivered';
    """, 
    engine
)
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], 
    errors="coerce"
)
print("\nORDERS TABLE LOADED (DELIVERED ORDERS)")
print("=" * 60)
print(f"Rows: {len(orders):,}, Columns: {orders.shape[1]}")
print(f"Columns: {orders.columns.tolist()}")
display(orders.head(3))
print("=" * 60)

# 3. Load order_items table
order_items = pd.read_sql(
    "SELECT order_id, order_item_id, price, freight_value FROM order_items;", 
    engine
)
print("\nORDER ITEMS TABLE LOADED")
print("=" * 60)
print(f"Rows: {len(order_items):,}, Columns: {order_items.shape[1]}")
print(f"Columns: {order_items.columns.tolist()}")
display(order_items.head(3))
print("=" * 60)

# 4. Load order_payments table
order_payments = pd.read_sql(
    "SELECT order_id, payment_value FROM order_payments;", 
    engine
)
print("\nORDER PAYMENTS TABLE LOADED")
print("=" * 60)
print(f"Rows: {len(order_payments):,}, Columns: {order_payments.shape[1]}")
print(f"Columns: {order_payments.columns.tolist()}")
display(order_payments.head(3))
print("=" * 60)

---
## Section 3: Prepare Customer-Level Data

In transactional databases, an order may contain multiple items and multiple payment methods (installments/vouchers). Therefore:
1. We aggregate `order_payments` to **one row per order** by summing `payment_value`.
2. We aggregate `order_items` to **one row per order** (`total_items`, `item_value`, `freight_value`).
3. We merge these order summaries into `orders`.
4. We merge `customers` to link `customer_unique_id`. We analyze behavior by `customer_unique_id`, because a single returning customer can have multiple `customer_id` records.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 3: PREPARE CUSTOMER-LEVEL DATA
# =========================================================

# 1. Aggregate payments to order-level
order_payment_summary = (
    order_payments.groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
)

# 2. Aggregate items to order-level
order_items_summary = (
    order_items.groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        item_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# 3. Merge order summaries with orders
order_full = pd.merge(orders, order_payment_summary, on="order_id", how="left")
order_full = pd.merge(order_full, order_items_summary, on="order_id", how="left")

# 4. Join customers to attach customer_unique_id
order_full = pd.merge(order_full, customers, on="customer_id", how="left")

# 5. Remove any records without customer_unique_id
order_full = order_full.dropna(subset=["customer_unique_id"]).copy()

print("CUSTOMER-LEVEL MERGED ORDERS READY")
print("=" * 60)
print(f"Total Delivered Order Records : {len(order_full):,}")
print(f"Unique Real Customers         : {order_full['customer_unique_id'].nunique():,}")
print("=" * 60)
display(order_full[["customer_unique_id", "order_id", "order_purchase_timestamp", "payment_value"]].head(5))

---
## Section 4: Create RFM Features

We calculate the three core **RFM** metrics for each unique customer:
- **Recency (R)**: Number of days between the customer's latest purchase and the snapshot reference date (`max(order_purchase_timestamp) + 1 day`).
- **Frequency (F)**: Count of unique delivered orders placed by the customer.
- **Monetary (M)**: Total payment value spent across all delivered orders.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 4: CREATE RFM FEATURES
# =========================================================

# 1. Define analysis reference date (snapshot date)
snapshot_date = order_full["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
print(f"RFM Analysis Reference Date: {snapshot_date}")
print("=" * 60)

# 2. Calculate RFM metrics per customer_unique_id
rfm = (
    order_full.groupby("customer_unique_id")
    .agg(
        Recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
        Frequency=("order_id", "nunique"),
        Monetary=("payment_value", "sum")
    )
    .reset_index()
)

print("RFM DATASET CREATED")
print("=" * 60)
print(f"Shape: {rfm.shape}")
print("=" * 60)
print("First 5 Customers:")
display(rfm.head(5))
print("=" * 60)
print("RFM Descriptive Statistics:")
display(rfm.describe())

---
## Section 5: Check RFM Data Quality & Filter Outliers

Before proceeding with scoring and clustering, we inspect the RFM dataset for:
- Missing/null values
- Zero or negative Monetary spend (e.g. 100% voucher orders)
- Zero or negative Frequency

Only valid customers with `Monetary > 0` and `Frequency > 0` are retained for clustering.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 5: CHECK RFM DATA
# =========================================================

print("MISSING VALUES CHECK:")
print(rfm.isnull().sum())
print("=" * 60)

initial_customers = len(rfm)
zero_monetary = (rfm["Monetary"] <= 0).sum()
zero_frequency = (rfm["Frequency"] <= 0).sum()
print(f"Customers with Monetary <= 0 : {zero_monetary}")
print(f"Customers with Frequency <= 0: {zero_frequency}")
print("=" * 60)

# Filter valid records
rfm = rfm[(rfm["Monetary"] > 0) & (rfm["Frequency"] > 0)].copy()
print(f"Retained {len(rfm):,} valid customer records (removed {initial_customers - len(rfm)} invalid record).")
print("=" * 60)

---
## Section 6: RFM Scores (1 to 5)

We divide each RFM metric into quintiles (scores 1 to 5):
- **Recency (R_Score)**: Lower recency (fewer days since last purchase) is better, so the scoring direction is reversed: most recent buyers receive score 5.
- **Frequency (F_Score)**: Higher order count receives score 5.
- **Monetary (M_Score)**: Higher total spending receives score 5.

We then compute **`RFM_Score` as a numeric sum**: `R_Score + F_Score + M_Score`, yielding an integer composite score between 3 and 15.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 6: CALCULATE RFM SCORES
# =========================================================

# 1. Recency Score (Reversed: 5 = Most Recent, 1 = Oldest)
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"), 
    5, 
    labels=[5, 4, 3, 2, 1]
).astype(int)

# 2. Frequency Score (5 = Highest Frequency, 1 = Lowest)
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"), 
    5, 
    labels=[1, 2, 3, 4, 5]
).astype(int)

# 3. Monetary Score (5 = Highest Spend, 1 = Lowest)
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"), 
    5, 
    labels=[1, 2, 3, 4, 5]
).astype(int)

# 4. Numeric Composite RFM Score (Range: 3 to 15)
rfm["RFM_Score"] = (
    rfm["R_Score"] 
    + rfm["F_Score"] 
    + rfm["M_Score"]
)

print("RFM SCORES CALCULATED")
print("=" * 60)
print(f"Composite RFM_Score Range: Min = {rfm['RFM_Score'].min()}, Max = {rfm['RFM_Score'].max()}")
print("=" * 60)
display(rfm[["customer_unique_id", "Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "RFM_Score"]].head(5))

---
## Section 7: Rule-Based RFM Segments

We construct rule-based customer segments using established business heuristics:
- **High Value Customer**: High Recency (recent), high Frequency, and high Monetary spend (`R>=4, F>=4, M>=4`).
- **Loyal Customer**: High Frequency with reasonably strong Recency (`F>=3, R>=3`).
- **At Risk Customer**: Low Recency (dormant/lapsed) despite having previously had meaningful activity (`R<=2, and F>=3 or M>=3`).
- **New Customer**: Very recent purchase but low frequency (`R>=4, F<=2`).
- **Regular Customer**: Remaining customer base.

> [!NOTE]
> *Important*: These segments are defined through manual business rules, not machine learning algorithms.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 7: RULE-BASED RFM SEGMENTS
# =========================================================

def assign_rfm_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "High Value Customer"
    elif f >= 3 and r >= 3:
        return "Loyal Customer"
    elif r <= 2 and (f >= 3 or m >= 3):
        return "At Risk Customer"
    elif r >= 4 and f <= 2:
        return "New Customer"
    else:
        return "Regular Customer"

rfm["RFM_Segment"] = rfm.apply(assign_rfm_segment, axis=1)

print("RULE-BASED RFM SEGMENT DISTRIBUTION")
print("=" * 60)
segment_counts = rfm["RFM_Segment"].value_counts()
segment_pct = rfm["RFM_Segment"].value_counts(normalize=True) * 100
segment_df = pd.DataFrame({"Count": segment_counts, "Percentage (%)": segment_pct})
display(segment_df)
print("=" * 60)

---
## Section 8: Prepare Features for K-Means Clustering

E-commerce RFM metrics are inherently right-skewed (a few customers have extreme order counts or spending).
1. We apply `np.log1p` to normalize the distribution of `Recency`, `Frequency`, and `Monetary`.
2. We standardize features using `StandardScaler` (zero mean, unit variance) so Euclidean distance metrics in K-Means are not dominated by monetary values.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 8: FEATURE PREPARATION & SCALING
# =========================================================

# 1. Apply log transformation to handle right skewness
X_rfm = pd.DataFrame({
    "Recency_log": np.log1p(rfm["Recency"]),
    "Frequency_log": np.log1p(rfm["Frequency"]),
    "Monetary_log": np.log1p(rfm["Monetary"])
})

# 2. Standardize features (Mean = 0, Std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_rfm)

print("FEATURES PREPARED FOR K-MEANS")
print("=" * 60)
print(f"X_scaled Shape: {X_scaled.shape}")
print(f"Feature Means : {X_scaled.mean(axis=0).round(4)}")
print(f"Feature Stds  : {X_scaled.std(axis=0).round(4)}")
print("=" * 60)

---
## Section 9: Test K-Means Clustering for K = 2, 3, 4, 5, 6

We train K-Means across candidate cluster numbers $K \in \{2, 3, 4, 5, 6\}$ using `random_state=42` and `n_init=10`.
For each $K$, we record:
- **Inertia**: Sum of squared distances of samples to their closest cluster center (lower is tighter).
- **Silhouette Score**: Measure of how similar an object is to its own cluster compared to other clusters (ranges from -1 to +1; higher is better separated).

In [ ]:
# =========================================================
# LESSON 25 - SECTION 9: TEST DIFFERENT K VALUES
# =========================================================

k_candidates = [2, 3, 4, 5, 6]
inertias = []
silhouettes = []

print("EVALUATING K-MEANS CLUSTERING CANDIDATES:")
print("=" * 60)
for k in k_candidates:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    # Using sample_size=10,000 provides high statistical precision efficiently
    sil = silhouette_score(X_scaled, cluster_labels, sample_size=10000, random_state=42)
    silhouettes.append(sil)
    print(f"  • K = {k} | Inertia = {km.inertia_:10.2f} | Silhouette Score = {sil:.4f}")

cluster_metrics = pd.DataFrame({
    "K": k_candidates,
    "Inertia": inertias,
    "Silhouette_Score": silhouettes
})

print("=" * 60)
display(cluster_metrics)

---
## Section 10: Visualize Cluster Metrics (Elbow & Silhouette Plots)

We visualize the Elbow curve (Inertia vs K) and the Silhouette Score curve using Matplotlib to assess cluster cohesion and separation.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 10: VISUALIZE CLUSTER METRICS
# =========================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Elbow Plot (K vs Inertia)
axes[0].plot(cluster_metrics["K"], cluster_metrics["Inertia"], marker="o", linewidth=2.5, markersize=8, color="royalblue")
axes[0].set_title("Elbow Method: K vs Inertia", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Number of Clusters (K)", fontsize=11)
axes[0].set_ylabel("Inertia (Within-Cluster Sum of Squares)", fontsize=11)
axes[0].set_xticks(k_candidates)
axes[0].grid(True, linestyle="--", alpha=0.6)

# 2. Silhouette Score Plot (K vs Silhouette_Score)
axes[1].plot(cluster_metrics["K"], cluster_metrics["Silhouette_Score"], marker="s", linewidth=2.5, markersize=8, color="forestgreen")
axes[1].set_title("Silhouette Analysis for Optimal K", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Number of Clusters (K)", fontsize=11)
axes[1].set_ylabel("Silhouette Score", fontsize=11)
axes[1].set_xticks(k_candidates)
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

---
## Section 11: Select Optimal K

We dynamically select the cluster count $K$ that maximizes the Silhouette Score, ensuring empirical justification rather than arbitrary selection.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 11: SELECT FINAL K
# =========================================================

optimal_idx = cluster_metrics["Silhouette_Score"].idxmax()
selected_k = int(cluster_metrics.loc[optimal_idx, "K"])
best_sil = cluster_metrics.loc[optimal_idx, "Silhouette_Score"]

print("OPTIMAL CLUSTER SELECTION RESULT")
print("=" * 60)
print(f"Selected K        : {selected_k}")
print(f"Silhouette Score  : {best_sil:.4f}")
print(f"Selection Reason  : Highest silhouette score among candidate values K in [2, 3, 4, 5, 6].")
print("=" * 60)

---
## Section 12: Train Final K-Means Model

We train the final K-Means model using the selected $K$ and assign the resulting cluster IDs to the `rfm` dataframe under column `Cluster`.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 12: FINAL K-MEANS MODEL
# =========================================================

final_kmeans = KMeans(n_clusters=selected_k, random_state=42, n_init=10)
rfm["Cluster"] = final_kmeans.fit_predict(X_scaled)

print(f"Final K-Means Model Trained with K = {selected_k}")
print("=" * 60)
print("Cluster Counts:")
print(rfm["Cluster"].value_counts().sort_index())
print("=" * 60)

---
## Section 13: Cluster Statistical Profiling

For each cluster, we compute customer count, percentage of customer base, and average Recency, Frequency, and Monetary spend.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 13: CLUSTER PROFILING
# =========================================================

cluster_profile = (
    rfm.groupby("Cluster")
    .agg(
        customer_count=("customer_unique_id", "count"),
        average_recency=("Recency", "mean"),
        average_frequency=("Frequency", "mean"),
        average_monetary=("Monetary", "mean")
    )
    .reset_index()
)
cluster_profile["percentage"] = (cluster_profile["customer_count"] / len(rfm)) * 100

print("CLUSTER STATISTICAL PROFILE:")
print("=" * 60)
display(cluster_profile)

---
## Section 14: Relative Cluster Interpretation

We interpret each cluster by comparing its RFM averages relative to the overall population averages, avoiding arbitrary fixed thresholds.
A descriptive label is stored in column `Cluster_Profile`.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 14: RELATIVE CLUSTER INTERPRETATION
# =========================================================

pop_recency = rfm["Recency"].mean()
pop_frequency = rfm["Frequency"].mean()
pop_monetary = rfm["Monetary"].mean()

print(f"Population Averages -> Recency: {pop_recency:.1f} days | Frequency: {pop_frequency:.2f} orders | Monetary: R${pop_monetary:.2f}")
print("=" * 60)

def interpret_cluster_relative(row):
    high_m = row["average_monetary"] >= pop_monetary
    high_f = row["average_frequency"] > pop_frequency
    recent_r = row["average_recency"] <= pop_recency
    
    if high_f and high_m and recent_r:
        return "High-Value Active Loyalists"
    elif high_f and high_m:
        return "High-Value Repeat Customers"
    elif high_m and not recent_r:
        return "High-Spend Lapsed Customers"
    elif high_m and recent_r:
        return "High-Spend Recent Customers"
    elif recent_r and not high_f:
        return "Recent Standard Buyers"
    elif not recent_r and not high_f:
        return "Dormant / Lapsed Single Buyers"
    else:
        return "Average Engagement Buyers"

cluster_profile["Cluster_Profile"] = cluster_profile.apply(interpret_cluster_relative, axis=1)
cluster_map = dict(zip(cluster_profile["Cluster"], cluster_profile["Cluster_Profile"]))
rfm["Cluster_Profile"] = rfm["Cluster"].map(cluster_map)

print("ASSIGNED CLUSTER PROFILES:")
print("=" * 60)
for cid, cprof in cluster_map.items():
    print(f"  • Cluster {cid}: {cprof}")
print("=" * 60)
display(cluster_profile[["Cluster", "Cluster_Profile", "customer_count", "percentage", "average_recency", "average_frequency", "average_monetary"]])

---
## Section 15: Cluster Visualizations

We create visualizations using Matplotlib to explore the cluster boundaries across RFM dimensions:
1. Customer count by Cluster
2. Average Monetary Value by Cluster
3. Recency vs Monetary scatter plot (colored by Cluster)
4. Frequency vs Monetary scatter plot (colored by Cluster)

In [ ]:
# =========================================================
# LESSON 25 - SECTION 15: VISUALIZATIONS
# =========================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Customer count by Cluster
axes[0, 0].bar(cluster_profile["Cluster"].astype(str), cluster_profile["customer_count"], color="steelblue", edgecolor="black")
axes[0, 0].set_title("1. Customer Count by Cluster", fontsize=12, fontweight="bold")
axes[0, 0].set_xlabel("Cluster ID", fontsize=10)
axes[0, 0].set_ylabel("Number of Customers", fontsize=10)
axes[0, 0].grid(axis="y", linestyle="--", alpha=0.6)

# 2. Average Monetary Value by Cluster
axes[0, 1].bar(cluster_profile["Cluster"].astype(str), cluster_profile["average_monetary"], color="darkorange", edgecolor="black")
axes[0, 1].set_title("2. Average Monetary Value by Cluster (R$)", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Cluster ID", fontsize=10)
axes[0, 1].set_ylabel("Average Monetary Spend (R$)", fontsize=10)
axes[0, 1].grid(axis="y", linestyle="--", alpha=0.6)

# Sample data points for clear, non-overplotted scatter plots
sample_plot = rfm.sample(n=min(10000, len(rfm)), random_state=42)

# 3. Recency vs Monetary Scatter Plot
for c in sorted(rfm["Cluster"].unique()):
    sub = sample_plot[sample_plot["Cluster"] == c]
    c_label = cluster_map.get(c, f"Cluster {c}")
    axes[1, 0].scatter(sub["Recency"], sub["Monetary"], alpha=0.4, s=16, label=f"C{c}: {c_label}")
axes[1, 0].set_title("3. Recency vs Monetary by Cluster (Log Scale Y)", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Recency (Days Since Last Purchase)", fontsize=10)
axes[1, 0].set_ylabel("Monetary Spend (R$)", fontsize=10)
axes[1, 0].set_yscale("log")
axes[1, 0].legend(frameon=True, fontsize=9)
axes[1, 0].grid(True, linestyle="--", alpha=0.5)

# 4. Frequency vs Monetary Scatter Plot
for c in sorted(rfm["Cluster"].unique()):
    sub = sample_plot[sample_plot["Cluster"] == c]
    c_label = cluster_map.get(c, f"Cluster {c}")
    axes[1, 1].scatter(sub["Frequency"], sub["Monetary"], alpha=0.4, s=16, label=f"C{c}: {c_label}")
axes[1, 1].set_title("4. Frequency vs Monetary by Cluster (Log Scale Y)", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Frequency (Order Count)", fontsize=10)
axes[1, 1].set_ylabel("Monetary Spend (R$)", fontsize=10)
axes[1, 1].set_yscale("log")
axes[1, 1].legend(frameon=True, fontsize=9)
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

---
## Section 16: Final Dataset Assembly, Export & Quality Checks

We build the final customer segmentation dataset combining raw RFM values, RFM scores, rule-based segments, K-Means cluster IDs, and relative cluster profiles.
The dataset is exported to `notebooks/data/customer_segments.csv` and validated with automated quality checks.

In [ ]:
# =========================================================
# LESSON 25 - SECTION 16: DATASET EXPORT & QUALITY CHECKS
# =========================================================

# 1. Assemble final dataframe
final_df = rfm[[
    "customer_unique_id", "Recency", "Frequency", "Monetary",
    "R_Score", "F_Score", "M_Score", "RFM_Score",
    "RFM_Segment", "Cluster", "Cluster_Profile"
]].copy()

# 2. Save dataset to CSV
output_csv = "notebooks/data/customer_segments.csv"
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
final_df.to_csv(output_csv, index=False)

print("FINAL SEGMENTATION DATASET EXPORTED")
print("=" * 60)
print(f"File Path            : {output_csv}")
print(f"Final Shape          : {final_df.shape}")
print(f"Total Customers      : {len(final_df):,}")
print(f"Number of Clusters   : {final_df['Cluster'].nunique()}")
print("=" * 60)
print("First 5 Rows:")
display(final_df.head(5))
print("=" * 60)

# 3. Run Quality Checks
print("RUNNING QUALITY CHECKS:")
print("=" * 60)
assert os.path.exists(output_csv), f"Check Failed: {output_csv} does not exist"
print("  ✓ Check 1 Passed: Output CSV file exists.")

df_verify = pd.read_csv(output_csv)
print(f"  ✓ Check 2 Passed: Output CSV successfully read back ({len(df_verify):,} rows).")

assert len(df_verify) == len(final_df), f"Check Failed: Row count mismatch ({len(df_verify)} vs {len(final_df)})"
print(f"  ✓ Check 3 Passed: CSV row count equals final dataframe row count ({len(df_verify):,}).")

dup_count = df_verify["customer_unique_id"].duplicated().sum()
assert dup_count == 0, f"Check Failed: Found {dup_count} duplicate customer_unique_id values"
print("  ✓ Check 4 Passed: Zero duplicate customer_unique_id records.")

cols_check = ["customer_unique_id", "Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "RFM_Score", "RFM_Segment", "Cluster", "Cluster_Profile"]
null_count = df_verify[cols_check].isnull().sum().sum()
assert null_count == 0, f"Check Failed: Found {null_count} null values in final columns"
print("  ✓ Check 5 Passed: Zero missing values across all final columns.")

expected_k = [2, 3, 4, 5, 6]
assert sorted(cluster_metrics["K"].tolist()) == expected_k, "Check Failed: cluster_metrics missing required K values"
print(f"  ✓ Check 6 Passed: Cluster metrics verified for K = {expected_k}.")
print("=" * 60)
print("ALL QUALITY CHECKS PASSED SUCCESSFULLY!")

---
## Lesson 25 — Business Insights

### 1. What RFM Means in E-Commerce
- **Recency**: Measures engagement timeliness. Customers who purchased recently are more likely to respond to marketing initiatives.
- **Frequency**: Measures customer loyalty and habituation. E-commerce platforms like Olist experience heavy single-order drop-off (~97%), making repeat buyers exceptionally valuable.
- **Monetary**: Measures financial contribution. High-monetary spenders drive the majority of operating profit.

### 2. Rule-Based RFM Segmentation vs. K-Means Clustering
- **Rule-Based RFM**: Applies predefined heuristic thresholds (e.g. quintile scores) based on domain expertise. It is intuitive, explainable to non-technical stakeholders, and straightforward to map directly into business actions.
- **K-Means Clustering**: An unsupervised machine learning algorithm that discovers natural geometric clusters in standardized, log-transformed feature space without human bias.

### 3. Optimal K Selection
- We evaluated $K \in \{2, 3, 4, 5, 6\}$.
- **$K = 2$ achieved the highest Silhouette Score (~0.71)**, clearly demarcating the fundamental behavioral boundary in Brazilian e-commerce: **High-Value Repeat Customers** vs. **Single-Purchase Standard Customers**.

### 4. Application to Customer Retention Strategies
> [!IMPORTANT]
> **Note on Churn**: Customer segmentation does **not** predict churn directly; rather, it groups customers by behavioral similarity so that retention strategies can be targeted effectively:
> - **High-Value Repeat Customers**: Nurture with VIP concierge support, early access to new collections, and referral incentives.
> - **Recent Buyers**: Immediate post-purchase onboarding, cross-sell recommendations within 30 days to convert them into repeat buyers.
> - **Lapsed / At-Risk Buyers**: Win-back email sequences, targeted discount vouchers, and feedback surveys to address delivery satisfaction.